# Paris winter HDD options — temperature model, pricing and backtest

**Question.** How much should a buyer pay for protection against a cold winter in Paris?
An HDD call pays 20 € per heating degree day above a strike, where
HDD = Σ max(18 °C − T, 0) over November–March. Energy suppliers buy this kind of protection
because gas and power demand rise with cold.

**Steps.** (1) download 35 years of daily temperatures, (2) model them as trend + seasonality +
AR(p) with seasonal volatility, (3) check the residuals, (4) price the call by Monte Carlo and
compare with *burn analysis*, the market's simple historical method, (5) backtest the forecasts
out of sample.

> On Google Colab, run the first cell to get the code. Set `SYNTHETIC = True` to run offline on
> simulated data with known parameters.

In [ ]:
# Colab: get the code (uploaded zip, or the GitHub repository)
import os, sys
if not os.path.exists("src"):
    if os.path.exists("weather-derivatives.zip"):
        !unzip -q -o weather-derivatives.zip
    elif not os.path.exists("weather-derivatives"):
        !git clone -q https://github.com/AElSama/weather-derivatives
    %cd weather-derivatives
sys.path.insert(0, ".")
SYNTHETIC = False

## 1. Data

Open-Meteo historical archive (ERA5 reanalysis, Paris grid point). Market convention: daily temperature = (Tmax + Tmin) / 2, Feb 29 removed.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from src.data import load_or_download, synthetic_paris
from src.model import TemperatureModel
from src.pricing import HDDContract, simulate_season, mc_price, burn_analysis, backtest, realised_hdd

df = synthetic_paris() if SYNTHETIC else load_or_download("data/paris_daily.csv")
print(df["date"].iloc[0].date(), "->", df["date"].iloc[-1].date(), f"({len(df):,} days)")
df.tail()

## 2. Model

T(t) = trend + seasonal mean + X(t), with X an AR(p) process whose shock volatility σ(d) depends on the day of the year. The AR order is chosen by BIC.

In [ ]:
model = TemperatureModel.fit(df["date"], df["temp"])
d = model.diagnostics
print(f"Warming trend       : {model.trend_per_decade:+.2f} °C per decade")
print(f"AR order (BIC)      : {d['ar_order']}, coefficients {np.round(model.phi, 3)}")
print(f"Daily shock vol     : winter {np.sqrt(model.seasonal_variance(pd.DatetimeIndex(['2001-01-15']))[0]):.2f} °C, "
      f"summer {np.sqrt(model.seasonal_variance(pd.DatetimeIndex(['2001-07-15']))[0]):.2f} °C")
print(f"ACF lag 1           : {d['resid_acf'][1]:.3f} before, {d['z_acf'][1]:.3f} after the AR filter")
print(f"Standardised shocks : skew {d['z_skew']:.2f}, excess kurtosis {d['z_excess_kurtosis']:.2f}")

**How to read this.** A positive trend means today's winters are warmer than the average of the sample.
If the lag-1 autocorrelation drops to about 0 after the filter, the AR part captures the day-to-day
persistence. A non-zero skew or kurtosis means shocks are not exactly Gaussian: a limit of the model
to keep in mind (cold spells are often sharper than warm spells).

## 3. Pricing the next winter

Simulate the winter from today's state, compare with burn analysis (past winters replayed, raw and detrended).

In [ ]:
last = df["date"].iloc[-1]
season = last.year if last < pd.Timestamp(f"{last.year}-11-01") else last.year + 1
sims = simulate_season(model, df, season, 20_000, seed=7)
burn = burn_analysis(df, model, season, list(range(int(df["date"].dt.year.min()), season)))
strike = float(np.round(np.median(sims), -1))
T = (pd.Timestamp(f"{season + 1}-03-31") - last).days / 365.25
call = HDDContract(season, strike, tick=20.0)
price, se = mc_price(call, sims, maturity_years=T)
disc = np.exp(-0.03 * T)
print(f"Winter {season}-{season + 1}: HDD mean {sims.mean():.0f}, 90% range [{np.quantile(sims, .05):.0f}, {np.quantile(sims, .95):.0f}]")
print(f"Call, strike {strike:.0f}, 20 €/HDD")
print(f"  model            : {price:,.0f} € (± {1.96 * se:,.0f})")
print(f"  burn, raw        : {call.payoff(burn['hdd_raw']).mean() * disc:,.0f} €")
print(f"  burn, detrended  : {call.payoff(burn['hdd_detrended']).mean() * disc:,.0f} €")

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(sims, bins=60, density=True, alpha=0.8, label="model")
ax.axvline(burn["hdd_raw"].mean(), color="C1", label="raw burn mean")
ax.axvline(burn["hdd_detrended"].mean(), color="C2", label="detrended burn mean")
ax.legend(); ax.set_xlabel("Winter HDD"); plt.show()

**How to read this.** With a warming trend, raw burn analysis replays colder past winters and
**overprices** HDD calls. Detrending each past winter to today's climate brings it close to the model.
The remaining gap between model and detrended burn comes from conditioning on today's temperatures and
from the 35-year sample being short.

## 4. Out-of-sample backtest

Parameters frozen on data before November 2010, then each later winter is forecast from the temperatures observed up to October 31.

In [ ]:
bt, _ = backtest(df, 2010, list(range(2010, season)), n_paths=5000)
inside = ((bt.realised > bt.model_q05) & (bt.realised < bt.model_q95)).mean()
rmse = lambda c: np.sqrt(np.mean((bt[c] - bt.realised) ** 2))
print(f"Realised HDD inside the 90% interval: {inside:.0%} of {len(bt)} winters (target 90%)")
print(f"RMSE: model {rmse('model_mean'):.0f}, raw burn {rmse('burn_raw'):.0f}, detrended burn {rmse('burn_detrended'):.0f}")
bt.round(0)

**How to read this.** Coverage close to 90% means the model's uncertainty is realistic. The comparison of
errors tells whether modelling beats the simple historical average. A warning sign is a run of winters
all on the same side of the forecast: it usually means the trend was badly extrapolated, which is the
main model risk for weather derivatives.

## 5. All figures

`run.py` produces the full set of figures and `results.json`.

In [ ]:
!python run.py {"--synthetic" if SYNTHETIC else ""}
from IPython.display import Image, display
for f in ["temperature_fit", "seasonal_volatility", "residual_diagnostics", "hdd_distribution", "call_price_vs_strike", "backtest"]:
    display(Image(f"figures/{f}.png"))

## 6. Download the results (Colab)

With real data, `run.py` has filled the results section of `README.md`. This cell downloads the project with its figures, ready to upload to GitHub.

In [ ]:
!cd .. && zip -qr weather-derivatives-results.zip weather-derivatives -x "*/.git/*" "*/data/*" "*/__pycache__/*"
try:
    from google.colab import files
    files.download("../weather-derivatives-results.zip")
except ImportError:
    print("Not on Colab: the zip is in the parent folder.")